# Figure 6

In [ ]:
#%%
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import h5py 
import pathlib
import re
from scipy import optimize, signal
from sonata.circuit import File

recurrent_dict = {
    'w/o recurrent':'',
    'w/ recurrent':'_recurrent'}

CELL_TYPES = ['e5ET','i5Pvalb','i5Sst']

colors = ['lightgrey',(188/255,223/255,187/255)]

data_folder = "Data_files_for_Figure_6/"

## Functions

In [ ]:
def form_network(config_js, infer=False):
    if infer:
        # in this case, config_js is the file pat, so trim the file name, and guess
        # the main directory name
        # pick up the top directory
        net_name = config_js.split("/")[0]
        net = File(
            f"{net_name}/network/v1_nodes.h5", f"{net_name}/network/v1_node_types.csv"
        )
    else:
        # get the network structure out of the simulation
        node_files = [e["nodes_file"] for e in config_js["networks"]["nodes"]]
        node_type_files = [e["node_types_file"] for e in config_js["networks"]["nodes"]]
        net = File(node_files[0], node_type_files[0])
    return net

def pick_core(df, radius=200.0):
    """return if the neuron is at the core."""
    lateral = np.sqrt(df["x"] ** 2 + df["z"] ** 2)
    return df[lateral <= radius]

def get_spikes(config_js, infer=False):
    if infer:
        dir_name = str(pathlib.Path(config_js).parent)
        spike_file_name = f"{dir_name}/spikes.csv"
    else:
        spike_file_name = config_js["output"]["spikes_file_csv"]
    spike_df = pd.read_csv(spike_file_name, sep=" ", index_col=2)
    return spike_df

def identify_cell_type(pop_name: str):
    if pop_name.startswith("e"):
        return "Exc"
    else:
        # return the string after the first number
        return re.search(r"\d+(.*)", pop_name).groups()[0]

# this is destructive method (adds columns to v1df)
def determine_sort_position(v1df, sortby=None):
    if v1df["location"].iloc[0] == "Cortex":  # Old model
        layer = v1df["pop_name"].apply(lambda x: x[1])
        v1df["location"] = layer
    if sortby is not None:
        sorter = ["location", "Cell Type", sortby]
    else:
        sorter = ["location", "Cell Type"]
    reset_v1 = v1df.sort_values(sorter).reset_index()
    # reset_v1 = v1df.sort_values(["location", "Cell Type", "x"]).reset_index()
    # reset_v1 = v1df.sort_values(["location", "Cell Type"]).reset_index()
    sort_position = reset_v1.sort_values("index").index
    return sort_position

def get_fr(config_file,equal_length = False,radius=200.0,start_time=1.0):
    net = form_network(config_file, infer=True)
    spike_df = get_spikes(config_file, infer=True)
    v1df = net.nodes["v1"].to_dataframe()
    v1df = pick_core(v1df, radius=radius)
    v1df["node_ids"] = v1df["node_id"]
    spike_df = spike_df.merge(v1df[["node_ids", "node_type_id"]], on="node_ids")
    duration = np.max(spike_df.timestamps.values)/1e3
    spike_df_spon = spike_df[spike_df.timestamps <= start_time*1e3]
    v1df["spontaneous"] = spike_df_spon.value_counts("node_ids") / start_time
    v1df["spontaneous"] = v1df["spontaneous"].fillna(0)
    if equal_length:
        spike_df_evoked = spike_df[(spike_df.timestamps > start_time*1e3) & (spike_df.timestamps <= 2*start_time*1e3)]
        v1df["evoked"] = spike_df_evoked.value_counts("node_ids") / start_time
    else:
        spike_df_evoked = spike_df[spike_df.timestamps > start_time*1e3]
        v1df["evoked"] = spike_df_evoked.value_counts("node_ids") / (duration-start_time)
    v1df["evoked"] = v1df["evoked"].fillna(0)
    v1df["delta_fr"] = v1df["evoked"] - v1df["spontaneous"]
    v1df['evoked_percent'] = (v1df["evoked"] - v1df["spontaneous"])/v1df['spontaneous']
    return v1df

def get_relative_angle_group(v1df,target,target_type = 'angle',binsize = 10):
    binedge = 180+binsize/2
    x_bins = np.arange(-binedge,binedge+binsize,binsize)
    if target_type == 'angle':
        target_angle = target
    elif target_type == 'neuron':
        target_angle = v1df.loc[v1df['node_id']==target,'tuning_angle'].values[0]
    v1df['relative_tuning_angle'] = v1df['tuning_angle'].values - target_angle
    v1df['relative_tuning_angle'] = v1df['relative_tuning_angle'].apply(lambda x: x if abs(x)<=180 else x-np.sign(x)*360)
    v1df['relative_angle_group'] = v1df['relative_tuning_angle'].apply(lambda x: x_bins[np.where(x>x_bins)[0][-1]]+binsize/2)
    return v1df, x_bins

def gaussian(x, a, b, c):
    return a * np.exp(-(b*x**2)/2) + c

def fit_gaussian(row,p0=None):
    x = row['relative_angle_group']
    y = row['mean']
    x_ = x/np.max(x)
    params, _ = optimize.curve_fit(gaussian, x_, y,p0)
    fit_y_ = gaussian(x_, params[0], params[1],params[2])
    if params[1]>=0:
        s = np.sqrt(1/params[1])*np.max(x)
    else:
        s = np.nan
    return s,fit_y_,params

def ps_for_MUA_df(row,bins=np.linspace(500,3000,501),nperseg=100):
    spikes = row['timestamps']
    n_neuron = row['n_neuron']
    (counts, bins) = np.histogram(spikes, bins=bins)
    binsize = bins[1]-bins[0]
    mua=counts/(n_neuron*binsize/1e3) 
    f, s = signal.welch(mua, fs=1/binsize*1e3, nperseg = nperseg, scaling='spectrum')
    s_norm = s/np.sum(s)
    return f,s,s_norm

def get_ps(config_file,bins=np.linspace(1000,10000,3601),nperseg=100):
    net = form_network(config_file, infer=True)
    spike_df = get_spikes(config_file, infer=True)

    v1df = net.nodes["v1"].to_dataframe()
    v1df = pick_core(v1df, radius=200)
    v1df["Cell Type"] = v1df["pop_name"].apply(identify_cell_type)
    v1df["Sort Position"] = determine_sort_position(v1df)

    spike_df = spike_df.loc[spike_df.index.isin(v1df.index)]
    spike_df["Sorted ID"] = v1df["Sort Position"].loc[spike_df.index]
    spike_df["Cell Type"] = v1df["Cell Type"].loc[spike_df.index]
    spike_df["pop_name"] = v1df["pop_name"].loc[spike_df.index]
    spike_df.reset_index(inplace=True)

    # get population activity for each cell type
    MUA_df = spike_df.groupby(['pop_name'])['timestamps'].agg(list).to_frame()
    MUA_df['n_neuron'] = spike_df.groupby(['pop_name'])['node_ids'].nunique()
    MUA_df.reset_index(inplace=True)

    # in this example, binsize = (4000-1000)/1200 = 2.5ms, segment size = 2.5*100 = 250ms
    MUA_df[['f','power','power_norm']] = MUA_df.apply(ps_for_MUA_df,bins=bins,nperseg=nperseg,axis=1,result_type='expand')
    f = MUA_df.loc[0,'f']

    # # average over trials
    # power_df = MUA_df.groupby(['layer','cell_type']).agg({'power_norm':list})
    # power_df.reset_index(inplace=True)
    # power_df['mean'] = power_df['power_norm'].apply(np.mean,axis=0)
    # power_df['std'] = power_df['power_norm'].apply(np.std,axis=0)
    # power_df.drop(columns=['power_norm'],inplace=True)

    return MUA_df

## Panel a - Gaussian input weights

In [ ]:
# %% Gaussian input weights
target = 90
x = np.linspace(0,360,300)
dx = x-target
dx[dx>180] = dx[dx>180]-360
dx[dx<-180] = dx[dx<-180]+360
sigma=60
plt.figure(figsize=(4,3),dpi=100)
plt.plot(x,np.exp(-(dx**2)/(2*sigma**2)),'k',linewidth=2)
plt.axvline(target,color='k',linestyle='--',alpha = 0.5)
plt.xlim([0,360])
plt.xticks([0,180,360])
plt.yticks([])
plt.xlabel('preferred angle\n'+r'$\theta_i$ [degrees]')
plt.ylabel('input weight')
plt.title('target angle\n'+r'$\theta_{target}$ = 90 [degrees]')
sns.despine()

## Panel b - 3D visualization of the model

In [ ]:
#%% load nodes and positions
folder = data_folder+'l5_circuit/network/'

file = 'v1_nodes.h5'
with h5py.File(folder+file,'r') as f_nodes:
    node_type_ids = f_nodes['nodes/v1/node_type_id'][()].astype(int)
    nodes_df = pd.DataFrame({
        'node_type_id':f_nodes['nodes/v1/node_type_id'][()].astype(int),
        'x':f_nodes['nodes/v1/0/x'][()],
        'y':f_nodes['nodes/v1/0/y'][()],
        'z':f_nodes['nodes/v1/0/z'][()],
        'tuning_angle':f_nodes['nodes/v1/0/tuning_angle'][()],
    })
# add pop_name
file = 'v1_node_types.csv'
v1_node_type_df = pd.read_csv(folder+file,sep=' ')
nodes_df = pd.merge(nodes_df,v1_node_type_df[['node_type_id','pop_name']],on='node_type_id',copy=False)
nodes_df.index.names = ['node_id']
nodes_df.reset_index(inplace=True)

nodes_df['radius'] = np.sqrt(nodes_df['x']**2+nodes_df['z']**2)

In [ ]:
#%% plot 3d network
pop_names = ["L5 ET", "L5 PV", "L5 SST"]
node_colors = ["tab:blue", "tab:purple", "tab:olive"]
# in nodes_df, for pop_name, change e5ET to L5 ET, etc.
nodes_df['pop_name'] = nodes_df['pop_name'].apply(lambda x: x.replace('e5ET','L5 ET'))
nodes_df['pop_name'] = nodes_df['pop_name'].apply(lambda x: x.replace('i5Pvalb','L5 PV'))
nodes_df['pop_name'] = nodes_df['pop_name'].apply(lambda x: x.replace('i5Sst','L5 SST'))

color_dict = dict(zip(pop_names, node_colors))
nodes_df['color'] = nodes_df['pop_name'].apply(lambda x: color_dict[x])

plt.rcParams['grid.color'] = (0.5, 0.5, 0.5, 0.3)

fig = plt.figure(figsize=(4,4),dpi=300)
ax = fig.add_subplot(111, projection='3d')
# ax.grid(False)
# ax.w_xaxis.set_pane_color((0.5, 0.5, 0.5, 0.05))
# ax.w_yaxis.set_pane_color((0.5, 0.5, 0.5, 0.05))
# ax.w_zaxis.set_pane_color((0.5, 0.5, 0.5, 0.05))
for i,pop_name in enumerate(pop_names):
    x,y,z,r,c = nodes_df[nodes_df['pop_name']==pop_name][['x','y','z','radius','color']].values.T
    ax.scatter(x[r<200],z[r<200],y[r<200],s=5,c=c[r<200],alpha=0.5,edgecolors='none',label=pop_name)
    r_range = (r>200) & (r<=300)
    ax.scatter(x[r_range],z[r_range],y[r_range],s=5,c=c[r_range],alpha=0.1,edgecolors='none')
ax.legend()
ax.set_xlabel(r'x [$\mu$m]')
ax.set_ylabel(r'z [$\mu$m]')
ax.set_zlabel(r'y [$\mu$m]')
ax.set_zticks([-500,-600])
ax.set_xticks([-300,-150,0,150,300])
ax.set_yticks([-300,-150,0,150,300])
ax.set_box_aspect((np.ptp(ax.get_xlim()), np.ptp(ax.get_ylim()), np.ptp(ax.get_zlim()))) 
ax.dist = 12

## Panel c - Firing rate distributions and Gaussian fits

In [ ]:
#%% plot fr and gaussian for original data
net = "l5_circuit"
# net = "l5_circuit/output_pss_w10"
cell_type = 'e5ET'
target_angle = 0
y_var = 'evoked'
sigma = 60
binsize = 10
# fig,ax = plt.subplots(nrows = 1,ncols=1,figsize = (3,3),dpi=300)
fig = plt.figure(figsize = (3,4.5),dpi=300)
fig.patch.set_facecolor('none')
df_all = pd.DataFrame()
for net_type,c in zip(recurrent_dict,colors):
    recurrent = recurrent_dict[net_type]
    config_file = data_folder+net+'/output_pssbkg_all_angles_sigma'+str(sigma)\
            +'/output_pssbkg'+recurrent+'_angle'+str(target_angle)\
            +'/config_pssbkg'+recurrent+'_angle'+str(target_angle)+'.json'
    print(config_file)
    v1df = get_fr(config_file, radius=200.0, start_time=1)
    v1df, x_bins = get_relative_angle_group(v1df,target_angle,target_type = 'angle',binsize=binsize)
    stats_df = v1df.groupby(['pop_name','relative_angle_group'])[y_var].agg(
        ['mean','std','count']).reset_index()
    cell_type_df = stats_df.groupby('pop_name').agg(list)
    p0 = [.7*stats_df['mean'].max(),(180/sigma)**2,.3*stats_df['mean'].max()]
    # fit gaussian for each cell type
    cell_type_df[['s','fit_y','params']] = cell_type_df.apply(fit_gaussian,args = (p0,),axis=1,result_type='expand')
    
    df = v1df.query('pop_name == "'+ cell_type +'"').reset_index()
    # plot data
    plt.scatter(df['relative_angle_group'],df[y_var],s=15,facecolor = c,alpha = 0.3,linewidth=0,label = net_type)

    # plot gaussian fit
    row = cell_type_df.loc[cell_type]
    fit_y = row['fit_y']
    x = np.sort(df['relative_angle_group'].unique())
    plt.plot(x, fit_y,color=c,linewidth = 3,label = net_type+' (Gaussian fit)')   

plt.legend()
plt.legend( loc  = 'upper center', bbox_to_anchor = (0.5, 1.5),  frameon = False)
plt.xticks([-180,0,180])
plt.xlabel('relative angle [degrees]')
plt.ylabel('spike rate [Hz]')
sns.despine()
# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()

## Panel d - Bar plot of standard deviation


In [ ]:
# %% plot std for original data
net = "l5_circuit"
s_all_df = pd.read_csv(data_folder+net+'/std_pssbkg_all_angles_sigma'+str(sigma)+'.csv',index_col=0)
df = s_all_df.query('pop_name == "'+ cell_type +'"').reset_index()
fig,ax = plt.subplots(figsize = (2,3),dpi=300)
fig.patch.set_facecolor('none')
sns.barplot(
    ax=ax,
    data = df,
    x ='recurrent',
    y = 's',
    errorbar = 'se',
    width = 0.5,
    palette = colors,
)
plt.xticks(rotation=10)
ax.axhline(sigma,color='k',linestyle='--')
ax.set_ylim(40,65)
ax.set_ylabel(r'$\sigma$ [degrees]')
ax.set_xlabel('')
sns.despine()

df_pivot = df.pivot(index = ['pop_name','target_angle'],columns = 'recurrent',values = 's')
df_pivot['diff'] = df_pivot['w/ recurrent']-df_pivot['w/o recurrent']
m = df_pivot['diff'].mean()
sem = df_pivot['diff'].sem()

## Panel e - Standard deviation of models with different inhibitory target fractions

In [ ]:
#%% plot change in std across exc frac
notune = ''
df_all = pd.DataFrame([])
for exc_frac in np.arange(0,1.1,0.1):
    net = "l5_circuit_exc_fraction_"+"{:.1f}".format(exc_frac)
    sigma = 60
    s_all_df = pd.read_csv(data_folder+net+'/std_pssbkg_all_angles_sigma'+str(sigma)+notune+'.csv',index_col=0)
    df = s_all_df.pivot(index = ['pop_name','target_angle'],columns = 'recurrent',values = 's')
    df['diff'] = df['w/ recurrent']-df['w/o recurrent']
    df = df.reset_index()
    df['exc_frac'] = exc_frac
    df['inh_frac'] = 1-exc_frac
    df_all = pd.concat([df_all,df])

fig,ax = plt.subplots(figsize = (3,3),dpi=300)
fig.patch.set_facecolor('none')
df = df_all.query('pop_name == "e5ET"')
sns.lineplot(
    ax=ax,
    data = df,
    x = 'inh_frac',
    y = 'diff',
    errorbar = 'se',
    linewidth = 1,
    markers=True,
    marker='o',
    color='k',

    err_kws={'alpha':0.2,'edgecolor':None}
)

ax.scatter(1-0.085,m,marker='*',s=100,color='r')
# ax.errorbar(0.085,m,yerr=1.96*sem,fmt='o',color='k',capsize=5)
ax.set_xticks(np.arange(0,1.2,0.2))
ax.axhline(0,color='k',linestyle='--',alpha=0.5)
# ax.set_xlabel('fraction of L5 ET targets')
ax.set_xlabel('fraction of inhibitory targets')
# make ylabel to be delta sigma
ax.set_ylabel('$\Delta \sigma$ [degrees]')
sns.despine()

# Panel f - Normalized power spectrum of example simulations

In [ ]:
#%% oscillation of original data
net = "l5_circuit"
sigma = 60
target_angle = 0

fig,ax = plt.subplots(nrows = 1,ncols = 1,figsize = (3,3),dpi=300)
fig.patch.set_facecolor('none')   
for net_type,c in zip(recurrent_dict,colors):
    recurrent = recurrent_dict[net_type]
    config_file = data_folder+net+'/output_pssbkg_all_angles_sigma'+str(sigma)\
        +'/output_pssbkg'+recurrent+'_angle'+str(target_angle)\
        +'/config_pssbkg'+recurrent+'_angle'+str(target_angle)+'.json'
    MUA_df = get_ps(config_file,bins=np.linspace(1000,10000,3601),nperseg=200)
    # MUA_df = get_ps(config_file,bins=np.linspace(1000,10000,1801),nperseg=100)
    MUA_df['peak_freq'] = MUA_df.apply(lambda row: row['f'][row['power'].argmax()],axis=1)
    # MUA_df_spon = get_ps(config_file,bins=np.linspace(0,1000,401),nperseg=100)
    df = MUA_df[MUA_df['pop_name']==cell_type]
    f = df['f'].values[0]
    power = df['power'].values[0]/df['power'].values[0].max()
    ax.plot(f,power,color = c,label = net_type)
    ax.set_xlim(0,100)
    ax.set_xlabel('frequency [Hz]')
    ax.set_ylabel('normalized power [a.u.]')
    sns.despine()
fig.tight_layout()


## Panel g - Bar plot of peak frequency

In [ ]:
# %%
net = "l5_circuit"
sigma = 60
peak_all_df = pd.read_csv(data_folder+net+'/oscpeak_pssbkg_all_angles_sigma'+str(sigma)+'.csv')
df = peak_all_df.query('pop_name == "'+ cell_type +'"').reset_index()
fig,ax = plt.subplots(figsize = (2,3),dpi=300)
fig.patch.set_facecolor('none')
sns.barplot(
    ax=ax,
    data = df,
    x ='recurrent',
    y = 'peak_freq',
    width = 0.5,
    palette = colors,
)
plt.xticks(rotation=10)
ax.set_ylim(0,40)
ax.set_ylabel('peak frequency [Hz]')
ax.set_xlabel('')
sns.despine()

m_freq = df.groupby(['pop_name','recurrent'])['peak_freq'].mean()
print(m_freq )
m_FWHM = df.groupby(['pop_name','recurrent'])['FWHM'].mean()
print(m_FWHM )
df_pivot = df.pivot(index=['pop_name','target_angle'],columns='recurrent',values=['peak_freq','FWHM'])
df_pivot['peak_freq_diff'] = df_pivot['peak_freq']['w/ recurrent']-df_pivot['peak_freq']['w/o recurrent']
df_pivot['FWHM_diff'] = df_pivot['FWHM']['w/ recurrent']-df_pivot['FWHM']['w/o recurrent']
m = df_pivot['peak_freq_diff'].mean()
sem = df_pivot['peak_freq_diff'].sem()

## Panel h - Peak frequency and FWHM of models with different inhibitory target fractions

In [ ]:
#%% plot peak freq for all
notune = ""
sigma = 60
df_all = pd.DataFrame([])
for exc_frac in np.arange(0,1.1,0.1):
    net = "l5_circuit_exc_fraction_"+"{:.1f}".format(exc_frac)
    print(net)
    peak_df = pd.read_csv(data_folder+net+'/oscpeak_pssbkg_all_angles_sigma'+str(sigma)+notune+'.csv',index_col=0)
    peak_df['exc_frac'] = exc_frac
    peak_df['inh_frac'] = 1-exc_frac
    df_all = pd.concat([df_all,peak_df])

# make a pivot table to calculate the difference
df_pivot = df_all.pivot(index=['exc_frac','pop_name','target_angle'],columns='recurrent',values=['peak_freq','FWHM'])
df_pivot['peak_freq_diff'] = df_pivot['peak_freq']['w/ recurrent']-df_pivot['peak_freq']['w/o recurrent']
df_pivot['FWHM_diff'] = df_pivot['FWHM']['w/ recurrent']-df_pivot['FWHM']['w/o recurrent']
df_pivot.reset_index(inplace=True)
df = df_pivot[df_pivot['pop_name']=='e5ET']

### peak frequency

In [ ]:
# %% plot peak freq for all exc_frac
df = df_all.query('pop_name == "'+ cell_type +'"').reset_index()
df = df.query('recurrent == "w/ recurrent"').reset_index()
ig,ax = plt.subplots(figsize=(3,3),dpi=300)
fig.patch.set_facecolor('none')
sns.lineplot(
    ax=ax,
    data = df,
    # x = 'exc_frac',
    x = 'inh_frac',
    y = 'peak_freq',
    errorbar = 'se',
    # y = 'FWHM',
    linewidth = 1,
    markers=True,
    marker='o',
    color=colors[1],
    err_kws={'alpha':0.2,'edgecolor':None},
    # zorder=1
)
ax.scatter(1-0.085,m_freq.values[0],marker='*',s=100,color='r',zorder=2)
ax.axhline(m_freq.values[1],color='k',linestyle='--',alpha=0.5)
ax.set_xticks(np.arange(0,1.2,0.2))
# ax.set_xlabel('fraction of L5 ET targets')
ax.set_xlabel('fraction of inhibitory targets')
ax.set_ylabel('peak frequency [Hz]')
sns.despine()

### FWHM

In [ ]:
# %% plot FWHM for all exc_frac
df = df_all.query('pop_name == "'+ cell_type +'"').reset_index()
df = df.query('recurrent == "w/ recurrent"').reset_index()
ig,ax = plt.subplots(figsize=(3,3),dpi=300)
fig.patch.set_facecolor('none')
sns.lineplot(
    ax=ax,
    data = df,
    # x = 'exc_frac',
    x = 'inh_frac',
    y = 'FWHM',
    errorbar = 'se',
    linewidth = 1,
    markers=True,
    marker='o',
    color=colors[1],
    err_kws={'alpha':0.2,'edgecolor':None},
    # zorder=1
)
ax.scatter(1-0.085,m_FWHM.values[0],marker='*',s=100,color='r',zorder=2)
ax.axhline(m_FWHM.values[1],color='k',linestyle='--',alpha=0.5)
ax.set_xticks(np.arange(0,1.2,0.2))
# ax.set_xlabel('fraction of L5 ET targets')
ax.set_xlabel('fraction of inhibitory targets')
ax.set_ylabel('FWHM [Hz]')
sns.despine()